In [ ]:
# Cell 1 — Install (run once per runtime)
# ─────────────────────────────────────────────────────────────────────
# Base deps are already installed by pip install -e from the repo.
!pip install -q -e ".[train,data]"

In [ ]:
# Cell 2 — Authenticate and mount Drive
# ─────────────────────────────────────────────────────────────────────
import pathlib
import shutil

from google.colab import drive  # type: ignore

drive.mount("/content/drive")

# Copy .env if it exists on Drive (contains API keys etc).
env_src = pathlib.Path("/content/drive/MyDrive/alignforge/.env")
if env_src.exists():
    shutil.copy(env_src, "/content/alignforge/.env")

In [ ]:
# Cell 3 — Clone or update the repo
# ─────────────────────────────────────────────────────────────────────
import subprocess

repo_url = "https://github.com/<your-username>/alignforge.git"
repo_dir = "/content/alignforge"

if not pathlib.Path(repo_dir).exists():
    subprocess.run(["git", "clone", repo_url, repo_dir], check=True)
else:
    subprocess.run(["git", "-C", repo_dir, "pull"], check=True)

%cd /content/alignforge

In [ ]:
# Cell 4 — Build dataset (skip if already built and saved to Drive)
# ─────────────────────────────────────────────────────────────────────
# Check if dataset on Drive.
import pathlib
import shutil

drive_data = pathlib.Path("/content/drive/MyDrive/alignforge/data")
local_data = pathlib.Path("data")

if drive_data.exists() and not local_data.exists():
    shutil.copytree(drive_data, local_data)
    print("Restored dataset from Drive.")
else:
    # Build fresh.
    !alignforge data build --config configs/data/sft_dev_assistant.yaml --skip-embedding

In [ ]:
# Cell 5 — Dry run: inspect config and hardware
# ─────────────────────────────────────────────────────────────────────
# Run this BEFORE Cell 6 to verify the plan without loading the model.
!alignforge train sft --config configs/train/sft_qlora.yaml --model-config configs/model/qwen2_5_1_5b.yaml --dataset-hash PASTE_HASH_FROM_DATA_BUILD_OUTPUT --dry-run

In [ ]:
# Cell 6 — Train
# ─────────────────────────────────────────────────────────────────────
# Replace PASTE_HASH with the actual dataset hash from Cell 4 output.
DATASET_HASH = "PASTE_HASH_HERE"
DRIVE_SYNC = "/content/drive/MyDrive/alignforge"

!alignforge train sft --config configs/train/sft_qlora.yaml --model-config configs/model/qwen2_5_1_5b.yaml --dataset-hash {DATASET_HASH} --drive-sync {DRIVE_SYNC}

In [ ]:
# Cell 7 — Inspect results
# ─────────────────────────────────────────────────────────────────────
!alignforge registry list --kind sft

# View probe generations (how did the model's output evolve?).
import json
import pathlib

probe_file = sorted(pathlib.Path("artifacts").glob("sft-*/probe_generations.jsonl"))
if probe_file:
    with probe_file[-1].open() as f:
        entries = [json.loads(label) for label in f]
    print(f"Probe entries: {len(entries)} across {len({e['step'] for e in entries})} checkpoints")
    # Show the last probe for each prompt.
    from collections import defaultdict

    by_prompt = defaultdict(list)

    for e in entries:
        by_prompt[e["prompt"]].append(e)

    for prompt, history in by_prompt.items():
        last = history[-1]
        print(f"\n[Step {last['step']}] Q: {prompt}")
        print(f"A: {last['response'][:300]}")